In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# -----------------------------
# CONFIG
# -----------------------------
BATCH_SIZE = 16
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(device)

# -----------------------------
# LOAD DATA
# -----------------------------
X_lig = np.load("data/processed_features/X_ligand.npy", mmap_mode='r')
X_prot = np.load("data/processed_features/X_protein.npy", mmap_mode='r')
y = np.load("data/processed_features/y.npy", mmap_mode='r')

# -----------------------------
# SPLIT
# -----------------------------
Xl_train, Xl_test, Xp_train, Xp_test, y_train, y_test = train_test_split(
    X_lig, X_prot, y, test_size=0.2, random_state=42
)

In [8]:
# Xl_train.shape()
X_prot

memmap([[ 0.01661635, -0.03018764,  0.00090213, ..., -0.04382138,
         -0.00518697, -0.01704268],
        [ 0.01661635, -0.03018764,  0.00090213, ..., -0.04382138,
         -0.00518697, -0.01704268],
        [ 0.04728722,  0.04910565,  0.01601403, ..., -0.04229094,
          0.01616114,  0.02428306],
        ...,
        [ 0.06604629,  0.08370882, -0.03349388, ..., -0.02958336,
         -0.03111628, -0.02133382],
        [ 0.10156471,  0.09728408, -0.01717847, ..., -0.05049216,
         -0.05006768, -0.05201894],
        [ 0.02670925, -0.01668411, -0.02484213, ..., -0.01539574,
         -0.01437376, -0.0419905 ]], shape=(1920867, 1024), dtype=float32)

In [ ]:

# -----------------------------
# DATASET
# -----------------------------
class DTIDataset(Dataset):
    def __init__(self, lig, prot, y):
        self.lig = lig
        self.prot = prot
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.lig[idx], dtype=torch.float32),
            torch.tensor(self.prot[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

train_loader = DataLoader(DTIDataset(Xl_train, Xp_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(DTIDataset(Xl_test, Xp_test, y_test), batch_size=BATCH_SIZE)

# -----------------------------
# MODEL
# -----------------------------
class DTIModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.ligand_net = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.protein_net = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, ligand, protein):
        l = self.ligand_net(ligand)
        p = self.protein_net(protein)

        x = torch.cat([l, p], dim=1)
        return self.fc(x).squeeze()

model = DTIModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# -----------------------------
# TRAIN
# -----------------------------
for epoch in range(10):
    model.train()
    total_loss = 0

    for ligand, protein, label in train_loader:
        ligand, protein, label = ligand.to(device), protein.to(device), label.to(device)

        pred = model(ligand, protein)
        loss = loss_fn(pred, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# -----------------------------
# EVALUATION
# -----------------------------
model.eval()
preds, true = [], []

with torch.no_grad():
    for ligand, protein, label in test_loader:
        ligand, protein = ligand.to(device), protein.to(device)

        pred = model(ligand, protein)

        preds.extend(pred.cpu().numpy())
        true.extend(label.numpy())

from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(true, preds))

print("Test RMSE:", rmse)